In [4]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parents[0]))
from market_inventory.universe import CoinUniverse, ProjectUniverse
from market_inventory.inventory import inventory_crypto_markets
from market_inventory.polymarket_clients import GammaClient


coin_universe = CoinUniverse.from_json(Path("/workspaces/Agents_v1/coins_universe.json"))
project_universe = ProjectUniverse.from_json(Path("/workspaces/Agents_v1/projects_universe.json"))
gamma = GammaClient()

df = inventory_crypto_markets(
    gamma=gamma,
    coin_universe=coin_universe,
    project_universe=project_universe,
    limit_events=500,
)

In [6]:
import pandas as pd
pd.set_option('display.max_colwidth', None)

In [7]:
import logging

logging.basicConfig(
    level=logging.INFO,   # set DEBUG for even more detail
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s"
)

In [9]:
from pipeline.historical_triage_runner import triage_dataframe_async

df_out = await triage_dataframe_async(
    df,
    max_rows=1,
    max_concurrency=1,
    row_timeout_s=120.0,
    total_timeout_s=300.0,
    log_every=1,
)

2026-02-08 15:06:34,303 | INFO | pipeline.historical_triage_runner | triage_dataframe_async start: rows=500 max_concurrency=1 only_source_nan_or_url=True max_rows=1
2026-02-08 15:06:34,308 | INFO | pipeline.historical_triage_runner | after filter: rows=241
2026-02-08 15:06:34,311 | INFO | pipeline.historical_triage_runner | after max_rows=1 cap: rows=1
2026-02-08 15:06:37,378 | INFO | httpx | HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"
2026-02-08 15:08:07,267 | INFO | httpx | HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2026-02-08 15:08:07,433 | INFO | pm_agents.historical_data_triage_agent | triage_market_row ok: market_id= relevance=yes feasibility=maybe paywall=possible (93.12s)
2026-02-08 15:08:07,433 | INFO | pipeline.historical_triage_runner | progress: 1/1 completed (1 ok, 0 errors)
2026-02-08 15:08:07,434 | INFO | pipeline.historical_triage_runner | triage completed: ok=1 errors=0
2026-02-08 15:08:07,437 | INF

2026-02-08 15:08:08,311 | INFO | httpx | HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"


In [11]:
df_out.to_csv("inspect_trage_v2.csv", index=False)